In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251118_143602"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path
import json

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "13_etl_dedup"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)

SNAPSHOT_ID = STAMP

BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Carregar dados de entrada (multifonte)
import pandas as pd
import glob

inputs = []

# Fonte principal do nb 12
f_parquet = os.path.join(RAW_PATH, "news_multisource.parquet")
if os.path.exists(f_parquet):
    dfp = pd.read_parquet(f_parquet)
    inputs.append(dfp)
    print("Carregado:", f_parquet, dfp.shape)

# Legado (se existir): noticias_real.csv
f_csv = os.path.join(RAW_PATH, "noticias_real.csv")
if os.path.exists(f_csv):
    dfc = pd.read_csv(f_csv)
    inputs.append(dfc)
    print("Carregado (legado):", f_csv, dfc.shape)

# Qualquer CSV extra solto (opcional)
for path in glob.glob(os.path.join(RAW_PATH, "noticias_real_*.csv")):
    try:
        dfx = pd.read_csv(path)
        inputs.append(dfx)
        print("Carregado (extra):", path, dfx.shape)
    except Exception as e:
        print("Falhou ao ler", path, "->", e)

assert len(inputs) > 0, "Nenhum dataset encontrado em data_raw. Rode o notebook 12 primeiro."
df_raw = pd.concat(inputs, ignore_index=True)
print("Dataset bruto consolidado:", df_raw.shape)
display(df_raw.head(3))

Carregado: C:\Users\ander\OneDrive\TCC_USP\data_raw\news_multisource.parquet (100, 7)
Carregado (legado): C:\Users\ander\OneDrive\TCC_USP\data_raw\noticias_real.csv (428, 4)
Carregado (extra): C:\Users\ander\OneDrive\TCC_USP\data_raw\noticias_real_dummy.csv (5, 5)
Dataset bruto consolidado: (533, 12)


,date,title,text,source,url,origin,uid,data,fonte,titulo,texto,clean_text
0,2025-09-19 07:00:00,Ibovespa bate novo recorde e encosta nos 146 m...,"<a href=""https://news.google.com/rss/articles/...",G1,https://news.google.com/rss/articles/CBMieEFVX...,google_rss,https://news.google.com/rss/articles/CBMieEFVX...,NaN,NaN,NaN,NaN,NaN
1,2025-09-21 07:00:00,Apenas 4 ações do Ibovespa pagam dividendos ac...,"<a href=""https://news.google.com/rss/articles/...",InfoMoney,https://news.google.com/rss/articles/CBMixAFBV...,google_rss,https://news.google.com/rss/articles/CBMixAFBV...,NaN,NaN,NaN,NaN,NaN
2,2025-09-30 07:00:00,Veja as 10 maiores altas do Ibovespa em setemb...,"<a href=""https://news.google.com/rss/articles/...",Valor Investe,https://news.google.com/rss/articles/CBMi_AFBV...,google_rss,https://news.google.com/rss/articles/CBMi_AFBV...,NaN,NaN,NaN,NaN,NaN


In [4]:
# 3. Normalizar esquema (COALESCE robusto: sem .str em DataFrame)
import pandas as pd, numpy as np, re

COLS = ["date","title","text","source","url","origin"]

def get_series(df, name):
    """Retorna uma Series mesmo se houver colunas duplicadas com o mesmo nome."""
    mask = (df.columns == name)
    if mask.sum() == 0:
        return pd.Series([None]*len(df), index=df.index, dtype='object')
    sub = df.loc[:, mask]
    if sub.shape[1] == 1:
        return sub.iloc[:, 0]
    # Coalesce por linha: 1º valor não-vazio/não-NaN
    return sub.apply(lambda row: next((x for x in row if pd.notna(x) and str(x).strip() != ""), None), axis=1)

def coalesce_synonyms(df, candidates):
    """Coalesce de vários nomes sinônimos -> 1 Series 'limpa'."""
    s = pd.Series([None]*len(df), index=df.index, dtype='object')
    for name in candidates:
        col = get_series(df, name)   # sempre Series
        mask = s.isna() | (s.astype(str).str.strip().isin(['', 'nan', 'None']))
        s = s.where(~mask, col)
    return s

# Sinônimos por coluna
date_candidates   = ["date","data","dt","pubDate","published","published_at","published_at_utc","publishedAt","Date"]
title_candidates  = ["title","Title","titulo","headline"]
text_candidates   = ["text","descricao","resumo","summary"]
source_candidates = ["source","SourceCommonName","source_name","fonte"]
url_candidates    = ["url","link","DocumentIdentifier"]
origin_candidates = ["origin"]

# Coalesce de cada campo
s_date   = coalesce_synonyms(df_raw, date_candidates)
s_title  = coalesce_synonyms(df_raw, title_candidates)
s_text   = coalesce_synonyms(df_raw, text_candidates)
s_source = coalesce_synonyms(df_raw, source_candidates)
s_url    = coalesce_synonyms(df_raw, url_candidates)
s_origin = coalesce_synonyms(df_raw, origin_candidates)

# Parse de datas (ISO, GDELT yyyymmddHHMMSS, fallback dd/MM/yyyy)
str_date = s_date.astype(str)

mask14 = str_date.str.fullmatch(r"\d{14}", na=False)              # ex.: 20250131094530
d1 = pd.to_datetime(str_date.where(mask14), format="%Y%m%d%H%M%S", errors="coerce")
d2 = pd.to_datetime(str_date.where(~mask14), errors="coerce")
date_parsed = d1.combine_first(d2)

if date_parsed.notna().sum() == 0 and str_date.str.contains(r"/").any():
    date_parsed = pd.to_datetime(str_date, format="%d/%m/%Y", errors="coerce")

# Monta dataframe final (Series -> strings limpas)
df = pd.DataFrame({
    "date":   date_parsed,
    "title":  s_title.astype(str).fillna("").str.strip(),
    "text":   s_text.astype(str).fillna("").str.strip(),
    "source": s_source.astype(str).fillna("").str.strip(),
    "url":    s_url.astype(str).fillna("").str.strip(),
    "origin": s_origin.astype(str).fillna("").str.strip()
})

# Remove inválidos e mostra amostra
before = df.shape[0]
df = df.dropna(subset=["date"]).copy()
df = df[df["title"].str.len() > 0].copy()
print(f"Removidas {before - df.shape[0]} linhas inválidas (sem data/sem título).")
print("Colunas finais:", df.columns.tolist())
display(df.head(3))

Removidas 433 linhas inválidas (sem data/sem título).
Colunas finais: ['date', 'title', 'text', 'source', 'url', 'origin']


,date,title,text,source,url,origin
0,2025-09-19 07:00:00,Ibovespa bate novo recorde e encosta nos 146 m...,"<a href=""https://news.google.com/rss/articles/...",G1,https://news.google.com/rss/articles/CBMieEFVX...,google_rss
1,2025-09-21 07:00:00,Apenas 4 ações do Ibovespa pagam dividendos ac...,"<a href=""https://news.google.com/rss/articles/...",InfoMoney,https://news.google.com/rss/articles/CBMixAFBV...,google_rss
2,2025-09-30 07:00:00,Veja as 10 maiores altas do Ibovespa em setemb...,"<a href=""https://news.google.com/rss/articles/...",Valor Investe,https://news.google.com/rss/articles/CBMi_AFBV...,google_rss


In [5]:
# 4. Normalizar URL (remover UTM/trailing e padronizar host)
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

def normalize_url(u: str) -> str:
    if not isinstance(u, str) or len(u.strip()) == 0 or u.strip().lower() in ["nan", "none"]:
        return ""
    try:
        p = urlparse(u.strip())
        # normalizar host
        netloc = p.netloc.lower()
        # filtrar query (remover utm_*, fbclid, gclid, etc.)
        keep_q = []
        for k,v in parse_qsl(p.query, keep_blank_values=False):
            if not (k.lower().startswith("utm_") or k.lower() in {"fbclid","gclid","mc_cid","mc_eid"}):
                keep_q.append((k,v))
        q = urlencode(keep_q, doseq=True)
        # remover barra final redundante
        path = p.path.rstrip("/")
        norm = urlunparse((p.scheme.lower(), netloc, path, "", q, ""))
        return norm
    except Exception:
        return u.strip()

df["url_norm"] = df["url"].apply(normalize_url)

In [6]:
# 5. Deduplicação (chave URL; fallback data+título)
has_url = df["url_norm"].str.len() > 0
df["uid"] = np.where(
    has_url,
    df["url_norm"],
    df["date"].dt.strftime("%Y-%m-%d") + "_" + df["title"].str[:80].str.replace(r"\W+", "_", regex=True)
)

before = df.shape[0]
df = df.drop_duplicates(subset=["uid"]).reset_index(drop=True)
removed = before - df.shape[0]
print(f"Duplicatas removidas: {removed}. Linhas finais: {df.shape[0]}")

Duplicatas removidas: 0. Linhas finais: 100

In [7]:
# 6. Qualidade (missing, origem, datas)
print("Faltantes por coluna (%):")
display((df[["title","text","source","url_norm","origin"]].isna().mean()*100).round(2))

print("Registros por origem:")
display(df["origin"].fillna("desconhecida").value_counts())

print("Faixa de datas:")
print(df["date"].min(), "→", df["date"].max())

Faltantes por coluna (%):


title       0.0
text        0.0
source      0.0
url_norm    0.0
origin      0.0
dtype: float64

Registros por origem:


origin
google_rss    100
Name: count, dtype: int64

Faixa de datas:
2025-09-19 07:00:00 → 2025-10-17 21:13:00


In [8]:
# 7. Salvar limpo + dicionário de dados + relatório
clean_parquet = os.path.join(PROC_PATH, "news_multisource_clean.parquet")
clean_csv     = os.path.join(PROC_PATH, "news_multisource_clean.csv")
dict_csv      = os.path.join(PROC_PATH, "dicionario_dados_news.csv")
report_json   = os.path.join(PROC_PATH, f"etl_report_{SNAPSHOT_ID}.json")

# salvar dados limpos
df_out = df[["date","title","text","source","url_norm","origin"]].rename(columns={"url_norm":"url"})
df_out.to_parquet(clean_parquet, index=False)
df_out.to_csv(clean_csv, index=False, encoding="utf-8")

# dicionário de dados
dic = pd.DataFrame([
    {"variavel":"date","tipo":"datetime64[ns]","descricao":"Data/hora de publicação em BRT (quando disponível)"},
    {"variavel":"title","tipo":"string","descricao":"Título/manchete da notícia"},
    {"variavel":"text","tipo":"string","descricao":"Resumo/descrição curta (se disponível)"},
    {"variavel":"source","tipo":"string","descricao":"Nome da fonte (ex.: Reuters Brasil)"},
    {"variavel":"url","tipo":"string","descricao":"URL normalizada (sem parâmetros de tracking)"},
    {"variavel":"origin","tipo":"string","descricao":"Origem técnica de coleta (newsapi, google_rss, gdelt, cvm, etc.)"},
], columns=["variavel","tipo","descricao"])
dic.to_csv(dict_csv, index=False, encoding="utf-8")

# relatório
report = {
    "notebook": NB_NAME,
    "snapshot_id": SNAPSHOT_ID,
    "inputs": {
        "raw_rows": int(df_raw.shape[0]),
        "raw_sources": list(set([c for c in ["news_multisource.parquet", "noticias_real.csv"] if os.path.exists(os.path.join(RAW_PATH, c))]))
    },
    "output_rows": int(df_out.shape[0]),
    "duplicates_removed": int(removed),
    "paths": {
        "clean_parquet": clean_parquet,
        "clean_csv": clean_csv,
        "data_dictionary": dict_csv
    }
}
with open(report_json, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("✅ Arquivos salvos:")
print(clean_parquet)
print(clean_csv)
print(dict_csv)
print(report_json)

✅ Arquivos salvos:
C:\Users\ander\OneDrive\TCC_USP\data_processed\news_multisource_clean.parquet
C:\Users\ander\OneDrive\TCC_USP\data_processed\news_multisource_clean.csv
C:\Users\ander\OneDrive\TCC_USP\data_processed\dicionario_dados_news.csv
C:\Users\ander\OneDrive\TCC_USP\data_processed\etl_report_20251118_143608.json


In [9]:
# 8. Resumo final (para colar no README de resultados)
from IPython.display import display, Markdown

rows_by_origin = df_out["origin"].fillna("desconhecida").value_counts().to_dict()
md = f"""
**{NB_NAME} – ETL & Dedup concluído**

- Linhas limpas: **{df_out.shape[0]}**
- Duplicatas removidas: **{removed}**
- Período: **{df_out['date'].min().date()} → {df_out['date'].max().date()}**
- Por origem: **{rows_by_origin}**

Arquivos:
- `data_processed/news_multisource_clean.parquet`
- `data_processed/news_multisource_clean.csv`
- `data_processed/dicionario_dados_news.csv`
- `data_processed/etl_report_{SNAPSHOT_ID}.json`
"""
display(Markdown(md))
print("Feito ✅")


**13_etl_dedup – ETL & Dedup concluído**

- Linhas limpas: **100**
- Duplicatas removidas: **0**
- Período: **2025-09-19 → 2025-10-17**
- Por origem: **{'google_rss': 100}**

Arquivos:
- `data_processed/news_multisource_clean.parquet`
- `data_processed/news_multisource_clean.csv`
- `data_processed/dicionario_dados_news.csv`
- `data_processed/etl_report_20251118_143608.json`


Feito ✅
